# Grad-CAM sobre InceptionV3 (modelo recomendado)

**Observación del revisor:** el Grad-CAM se aplicó sobre DenseNet121 (7º lugar, 96.70%) en vez de InceptionV3 (el modelo ganador y recomendado para despliegue, 98.18%).

**Corrección:** genera Grad-CAM sobre **InceptionV3**, manteniendo también el de DenseNet121 para comparación.

Entrena InceptionV3 (~15 min) y genera las visualizaciones. GPU T4.

In [ ]:
!pip install -q kaggle timm scikit-learn grad-cam
import torch
print('GPU:', torch.cuda.is_available())

In [ ]:
from google.colab import files
import os
print('Sube tu kaggle.json:')
uploaded = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
for fn in uploaded.keys(): os.rename(fn, '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p /content/data --unzip
print('Dataset listo')

In [ ]:
import shutil, random
from pathlib import Path
random.seed(42)
SRC=Path('/content/data/chest_xray')
ALL={'NORMAL':[],'PNEUMONIA':[]}
for split in ['train','test','val']:
    sd=SRC/split
    if not sd.exists(): continue
    for img in (sd/'NORMAL').glob('*.jpeg'): ALL['NORMAL'].append(img)
    for img in (sd/'PNEUMONIA').glob('*.jpeg'): ALL['PNEUMONIA'].append(img)
DEST=Path('/content/dataset_bin')
if DEST.exists(): shutil.rmtree(DEST)
for cls,imgs in ALL.items():
    imgs=imgs.copy(); random.shuffle(imgs)
    n=len(imgs); ntr=int(n*0.70); nv=int(n*0.15)
    for sn,si in {'train':imgs[:ntr],'val':imgs[ntr:ntr+nv],'test':imgs[ntr+nv:]}.items():
        od=DEST/sn/cls; od.mkdir(parents=True,exist_ok=True)
        for img in si: shutil.copy(img,od/img.name)
print('Partición lista')

In [ ]:
# Entrenar InceptionV3 (mismo protocolo que el experimento principal)
import timm, numpy as np, time
import torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import accuracy_score

device=torch.device('cuda')
torch.manual_seed(42); np.random.seed(42)
IMG=299  # InceptionV3 requiere 299x299

train_tf=transforms.Compose([transforms.Resize((IMG,IMG)),transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(10),transforms.ColorJitter(brightness=0.15,contrast=0.15),
    transforms.ToTensor(),transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
eval_tf=transforms.Compose([transforms.Resize((IMG,IMG)),transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

tr=DataLoader(datasets.ImageFolder('/content/dataset_bin/train',transform=train_tf),batch_size=32,shuffle=True,num_workers=2)
va=DataLoader(datasets.ImageFolder('/content/dataset_bin/val',transform=eval_tf),batch_size=32,num_workers=2)
te=DataLoader(datasets.ImageFolder('/content/dataset_bin/test',transform=eval_tf),batch_size=32,num_workers=2)
classes=datasets.ImageFolder('/content/dataset_bin/test').classes

model=timm.create_model('inception_v3',pretrained=True,num_classes=2,aux_logits=False).to(device)
crit=nn.CrossEntropyLoss(); opt=optim.AdamW(model.parameters(),lr=1e-4,weight_decay=0.01)
sch=optim.lr_scheduler.CosineAnnealingLR(opt,T_max=10)
best,bstate,noimp=0.0,None,0

for ep in range(10):
    t0=time.time(); model.train()
    for x,y in tr:
        x,y=x.to(device),y.to(device)
        opt.zero_grad(); crit(model(x),y).backward(); opt.step()
    sch.step(); model.eval(); vp,vt=[],[]
    with torch.no_grad():
        for x,y in va:
            vp.extend(model(x.to(device)).argmax(1).cpu().numpy()); vt.extend(y.numpy())
    vacc=accuracy_score(vt,vp)
    print(f'  Ep {ep+1}/10 val={vacc:.4f} ({time.time()-t0:.0f}s)')
    if vacc>best: best=vacc; bstate={k:v.cpu().clone() for k,v in model.state_dict().items()}; noimp=0
    else:
        noimp+=1
        if noimp>=4: print(f'  Early stop ep {ep+1}'); break
if bstate: model.load_state_dict(bstate)

# Exactitud en test
model.eval(); tp,tl=[],[]
with torch.no_grad():
    for x,y in te:
        tp.extend(model(x.to(device)).argmax(1).cpu().numpy()); tl.extend(y.numpy())
print(f'\n>> InceptionV3 TEST: {accuracy_score(tl,tp)*100:.2f}%')

In [ ]:
# Grad-CAM sobre InceptionV3
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
import matplotlib.pyplot as plt
from PIL import Image

model.eval()
# Última capa convolucional de InceptionV3 en timm
target_layer = model.Mixed_7c
print('Capa objetivo:', type(target_layer).__name__)

# Ejemplos: 2 normales, 2 con neumonía
test_dir=Path('/content/dataset_bin/test')
ejemplos=[]
for cls in ['NORMAL','PNEUMONIA']:
    for im in sorted((test_dir/cls).glob('*.jpeg'))[:2]:
        ejemplos.append((im, cls))

cam = GradCAM(model=model, target_layers=[target_layer])
fig, axes = plt.subplots(len(ejemplos), 2, figsize=(9, 4.5*len(ejemplos)))

for i,(path,cls) in enumerate(ejemplos):
    img = Image.open(path).convert('RGB').resize((IMG,IMG))
    rgb = np.array(img)/255.0
    tensor = eval_tf(img).unsqueeze(0).to(device)
    pred = model(tensor).argmax(1).item()
    gray = cam(input_tensor=tensor, targets=[ClassifierOutputTarget(pred)])[0]
    vis = show_cam_on_image(rgb, gray, use_rgb=True)
    axes[i,0].imshow(rgb); axes[i,0].set_title(f'Original — Real: {cls}', fontsize=10); axes[i,0].axis('off')
    axes[i,1].imshow(vis); axes[i,1].set_title(f'Grad-CAM InceptionV3 — Predicho: {classes[pred]}', fontsize=10); axes[i,1].axis('off')

plt.tight_layout()
plt.savefig('/content/gradcam_inceptionv3.png', dpi=150, bbox_inches='tight')
plt.show()

files.download('/content/gradcam_inceptionv3.png')
print('\nPasame gradcam_inceptionv3.png')